In [0]:
# Install required libraries
# azure-storage-blob: connects to Azure Blob Storage
# openpyxl: required by pandas to read .xlsx Excel files
%pip install azure-storage-blob openpyxl

In [0]:
# Restart kernel after install to load new libraries
dbutils.library.restartPython()

## Connect & load FocusPOS raw file

In [0]:
# ============================================
# Connect to Azure Blob Storage
# Load FocusPOS raw Excel from bronze
# ============================================
from azure.storage.blob import BlobServiceClient
import pandas as pd
import io

# Connect securely via Key Vault — never hardcode credentials
storage_account_key = dbutils.secrets.get(scope="churchs-payroll-kv", key="storage-account-key")
blob_service_client = BlobServiceClient(
    account_url="https://churchspayrollstorage.blob.core.windows.net",
    credential=storage_account_key
)
print("Connected to Azure Blob Storage successfully!")

# Point to FocusPOS file in bronze container
blob_client = blob_service_client.get_blob_client(
    container="bronze",
    blob="focuspos/FocusPOS Input.xlsx"
)

# Download and read Excel without headers
blob_data = blob_client.download_blob().readall()
df_raw_focus = pd.read_excel(io.BytesIO(blob_data), header=None)

# Validation — check file loaded correctly
print("Raw file loaded! Shape:", df_raw_focus.shape)
assert df_raw_focus.shape[0] > 0, "ERROR: Raw file is empty!"
assert df_raw_focus.shape[1] > 0, "ERROR: No columns found!"
print("Validation passed — file has", df_raw_focus.shape[0], "rows and", df_raw_focus.shape[1], "columns")

## Parse hierarchy

In [0]:
# ============================================
# Parse FocusPOS hierarchy
# Structure: Store → Employee → Shift rows
# No standard headers — must detect row types manually
# ============================================

records = []
store = employee = payroll = position = None

# Track counts for logging
store_count = 0
employee_count = 0
skipped_rows = 0

for idx, row in df_raw_focus.iterrows():
    c2 = str(row[2]).strip() if pd.notna(row[2]) else ""
    c3 = str(row[3]).strip() if pd.notna(row[3]) else ""
    c5 = str(row[5]).strip() if pd.notna(row[5]) else ""

    # Store name row — identified by Churchs keyword
    if "Churchs" in c2:
        store = c2
        store_count += 1

    # Employee name and payroll ID row
    # ID1 = SSN (ignored), ID2 = payroll ID
    elif c3 and "ID1:" in c5:
        employee = c3
        payroll = str(row[15]).strip() if pd.notna(row[15]) else ""
        employee_count += 1

    # Shift data row — has valid date in column 5
    elif pd.notna(row[5]) and "2026" in str(row[5]):
        try:
            records.append({
                "StoreID":       store,
                "Employee_name": employee,
                "Payroll_ID":    payroll,
                "Position":      c3,
                "Shift_Date":    pd.to_datetime(row[5], errors="coerce"),
                "Clock_In":      row[10],
                "Clock_Out":     row[14],
                "Hours":         pd.to_numeric(row[17], errors="coerce"),
                "Rate":          pd.to_numeric(row[19], errors="coerce"),
                "Pay":           pd.to_numeric(row[21], errors="coerce"),
                "Total_Pay":     pd.to_numeric(row[24], errors="coerce"),
                "Declared_Tips": pd.to_numeric(row[32], errors="coerce")
            })
        except Exception as e:
            skipped_rows += 1
            print(f"Warning: Skipped row {idx} — {e}")

# Build silver DataFrame
df_silver_focus = pd.DataFrame(records)

# Fix week_end_date — Saturday of each shift week
df_silver_focus["week_end_date"] = df_silver_focus["Shift_Date"].apply(
    lambda d: (d + pd.Timedelta(days=(5 - d.weekday()) % 7)).date() if pd.notna(d) else None
)

# Fix Shift_Date to date only — remove time component
df_silver_focus["Shift_Date"] = df_silver_focus["Shift_Date"].dt.date

# Logs
print("\n=== Parse Summary ===")
print("Stores found:", store_count)
print("Employees found:", employee_count)
print("Shift records parsed:", len(records))
print("Rows skipped:", skipped_rows)

# Validation checks
print("\n=== Validation Checks ===")
assert len(df_silver_focus) > 0, "ERROR: No records parsed!"
assert df_silver_focus["StoreID"].notna().all(), "ERROR: Missing StoreID values!"
assert df_silver_focus["Employee_name"].notna().all(), "ERROR: Missing Employee names!"
assert df_silver_focus["Payroll_ID"].notna().all(), "ERROR: Missing Payroll IDs!"
print("All validation checks passed!")

print("\nUnique stores:", df_silver_focus["StoreID"].nunique())
print("Unique employees:", df_silver_focus["Payroll_ID"].nunique())
print("Date range:", df_silver_focus["Shift_Date"].min(), "to", df_silver_focus["Shift_Date"].max())
print("\nSample:")
print(df_silver_focus.head(5).to_string())

## Upload Cleaned FocusPOS Data to Silver Container

In [0]:
# ============================================
# Upload cleaned FocusPOS data
# to silver container as CSV
# ============================================

# Convert DataFrame to CSV in memory
csv_buffer = io.BytesIO()
df_silver_focus.to_csv(csv_buffer, index=False)
csv_buffer.seek(0)

# Upload to silver container
silver_client = blob_service_client.get_blob_client(
    container="silver",
    blob="focuspos/FocusPOS_Silver.csv"
)
silver_client.upload_blob(csv_buffer, overwrite=True)

# Validation — verify upload
print("FocusPOS Silver CSV uploaded successfully!")
print("Location: silver/focuspos/FocusPOS_Silver.csv")
print("Total records written:", len(df_silver_focus))
print("Columns written:", df_silver_focus.columns.tolist())
assert len(df_silver_focus) > 0, "ERROR: Nothing was written to silver!"
print("Upload validation passed!")